### Feature Encoding
Feature encoding is the process of converting categorical data into numerical values, so that machine learning algorithms can process the information.

In [1]:
# Importing necessary libraries
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

In [2]:
# Loading dataset
df = pd.read_csv('data/customer.csv')
X_train, X_test, y_train, y_test = train_test_split(df.loc[:,['review', 'education']], df.iloc[:,-1], test_size=0.2)

df.head()

,age,gender,review,education,purchased
0,30,Female,Average,School,No
1,68,Female,Poor,UG,No
2,70,Female,Good,PG,No
3,72,Female,Good,PG,No
4,16,Female,Average,UG,No


#### Ordinal Encoding

In [3]:
# Encoder
from sklearn.preprocessing import OrdinalEncoder

oe = OrdinalEncoder(
    categories=[['Poor','Average','Good'],['School','UG','PG']]
) # You have to mention the categories in the order of lower importance to higher otherwise encoder will automatically assign new categories as per they come into picture.
oe.fit(X_train)

,categories,"[['Poor', 'Average', ...], ['School', 'UG', ...]]"
,dtype,<class 'numpy.float64'>
,handle_unknown,'error'
,unknown_value,None
,encoded_missing_value,nan
,min_frequency,None
,max_categories,None


In [4]:
# Transformation
X_train = oe.transform(X_train)
X_test = oe.transform(X_test)
X_test[:5]

array([[2., 2.],
       [2., 1.],
       [2., 1.],
       [0., 1.],
       [0., 1.]])

In [5]:
# Attributes
print(oe.categories_)
print(oe.get_feature_names_out())

[array(['Poor', 'Average', 'Good'], dtype=object), array(['School', 'UG', 'PG'], dtype=object)]
['review' 'education']


In [6]:
# Inverse Encoding
oe.inverse_transform(np.array([[0,2]])) # The passing array must be in 2D

array([['Poor', 'PG']], dtype=object)

The handle_unknown parameter determines how the encoder should handle categories that were not seen during the training phase (i.e., new categories that appear during transformation). There are two possible values:

- 'error' (default): Raises an error when an unknown category is encountered during transformation.

- 'use_encoded_value': When this option is used, you must also specify an unknown_value parameter. Any unknown categories encountered during transformation will be encoded with this specified value. In your code example, unknown categories will be encoded as -1.

In [7]:
# Handling unknown value
oe = OrdinalEncoder(
    categories=[['Poor','Average','Good'], ['School','UG','PG']],
    handle_unknown='use_encoded_value', # The value is provided with unknown_value parameter - This will be applied for all the columns that have been encoded through this single encoding object. Possible values ("error", "use_encoded_value")
    unknown_value=-1 # Encoding value
)

# Splitting the dataset
X_train, X_test, y_train, y_test = train_test_split(df.loc[:,['review', 'education']], df.iloc[:,-1], test_size=0.2)

# Training the encoder
oe.fit(X_train)

,categories,"[['Poor', 'Average', ...], ['School', 'UG', ...]]"
,dtype,<class 'numpy.float64'>
,handle_unknown,'use_encoded_value'
,unknown_value,-1
,encoded_missing_value,nan
,min_frequency,None
,max_categories,None


##### Handling Rare Categories

In [8]:
# Creating dummy data
X = np.array([['dog'] * 5 + ['cat'] * 20 + ['rabbit'] * 10 + ['snake'] * 3 + ['horse'] * 2], dtype=object).T

# Frequencies of each category
pd.Series(X.ravel()).value_counts()

cat       20
rabbit    10
dog        5
snake      3
horse      2
Name: count, dtype: int64

In [9]:
# max_categories parameter
enc = OrdinalEncoder(max_categories=3) # The lowest (n - max_categories + 1) categories are considered as rare categories.
enc.fit(X)

,categories,'auto'
,dtype,<class 'numpy.float64'>
,handle_unknown,'error'
,unknown_value,None
,encoded_missing_value,nan
,min_frequency,None
,max_categories,3


In [10]:
# Rare categories
enc.infrequent_categories_

[array(['dog', 'horse', 'snake'], dtype=object)]

In [11]:
# Transforming
enc.transform(np.array([['cat', 'rabbit', 'snake', 'dog', 'horse']]).reshape(5,1)) # 'snake' and 'dog' encoded to hignest encoding value 3.

array([[0.],
       [1.],
       [2.],
       [2.],
       [2.]])

In [12]:
# min_frequency parameter
enc = OrdinalEncoder(min_frequency=4)# The categories whose frequency is less than or equal to 4 considered as rare categories
enc.fit(X)

,categories,'auto'
,dtype,<class 'numpy.float64'>
,handle_unknown,'error'
,unknown_value,None
,encoded_missing_value,nan
,min_frequency,4
,max_categories,None


In [13]:
# Rare categories
enc.infrequent_categories_

[array(['horse', 'snake'], dtype=object)]

In [14]:
# Transforming
enc.transform(np.array([['cat','rabbit','snake','dog','horse']]).reshape(5,1))

array([[0.],
       [2.],
       [3.],
       [1.],
       [3.]])

In [15]:
# Handling missing data
data = [['Cat'], [np.nan], ['Dog'], ['Fish'], [np.nan]]

encoder = OrdinalEncoder(encoded_missing_value=-1)
encoder.fit_transform(data)

array([[ 0.],
       [-1.],
       [ 1.],
       [ 2.],
       [-1.]])

---

#### Label Encoding

In [16]:
# Splitting the data
X_train, X_test, y_train, y_test = train_test_split(df.iloc[:,1:3], df.iloc[:,-1], test_size=0.2)
X_train.head()

,gender,review
41,Male,Good
5,Female,Average
48,Female,Good
1,Female,Poor
9,Male,Good


In [17]:
# Training the encoder
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder() # Since this encoder is used for unordered categorical features. No need to mention the categories and its order
le.fit(y_train)

LabelEncoder()

In [18]:
# Transforming
y_train = le.transform(y_train)
y_test = le.transform(y_test)
y_train

array([1, 1, 1, 0, 1, 1, 1, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0,
       1, 1, 0, 0, 1, 1, 1, 0, 0, 0, 1, 1, 1, 0, 1, 1, 0, 1])

In [19]:
# Encoded Categories
le.classes_

array(['No', 'Yes'], dtype=object)

In [20]:
# Reverse transformation
le.inverse_transform(np.array([1, 1, 0]))

array(['Yes', 'Yes', 'No'], dtype=object)

`Label Encoder` is neither capable of handling the unknown labels nor missing values

---

#### One Hot Encoding

In [21]:
# Loading and splitting the dataset
cars = pd.read_csv('data/cars.csv')

X = cars.iloc[:, [0, 2]]
y = cars.iloc[:, -1]
X_train,X_test,y_train,y_test = train_test_split(X, y, test_size=0.2, random_state=42)

cars.head()

,brand,km_driven,fuel,owner,selling_price
0,Maruti,145500,Diesel,First Owner,450000
1,Skoda,120000,Diesel,Second Owner,370000
2,Honda,140000,Petrol,Third Owner,158000
3,Hyundai,127000,Diesel,First Owner,225000
4,Maruti,120000,Petrol,First Owner,130000


In [22]:
# Unique categories in fuel feature
X_train['fuel'].unique()

array(['Petrol', 'Diesel', 'CNG', 'LPG'], dtype=object)

In [23]:
# Training the encoder
from sklearn.preprocessing import OneHotEncoder

ohe = OneHotEncoder(
    categories = [
        X_train['brand'].unique().tolist(),
        X_train['fuel'].unique().tolist()
    ],
    sparse_output = False, 
    dtype = np.int32
) # All the columns get's encoded seperately by single object
# sparse_output = False - Directly returns the encoded matrix
ohe.fit(X_train)

,categories,"[['Tata', 'Honda', ...], ['Petrol', 'Diesel', ...]]"
,drop,None
,sparse_output,False
,dtype,<class 'numpy.int32'>
,handle_unknown,'error'
,min_frequency,None
,max_categories,None
,feature_name_combiner,'concat'


In [24]:
# All categories
ohe.categories_

[array(['Tata', 'Honda', 'Hyundai', 'Maruti', 'Mahindra', 'Volkswagen',
        'Toyota', 'Force', 'Skoda', 'BMW', 'Fiat', 'Ford', 'Jaguar',
        'Renault', 'Jeep', 'Nissan', 'Chevrolet', 'Datsun',
        'Mercedes-Benz', 'Lexus', 'Mitsubishi', 'Volvo', 'Audi', 'Ashok',
        'Peugeot', 'Land', 'Ambassador', 'Isuzu', 'MG', 'Opel', 'Daewoo',
        'Kia'], dtype=object),
 array(['Petrol', 'Diesel', 'CNG', 'LPG'], dtype=object)]

In [25]:
# Reverse Encoding
ohe.inverse_transform(np.array([0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0.]).reshape(1,36))

array([['Hyundai', 'CNG']], dtype=object)

---

#### Dummy Variable Trap
When you have a categorical variable with n categories, you only need n-1 binary columns to represent it completely. This is because:
- If you have n categories and create n binary columns
- `The last column can always be perfectly predicted from the other n-1 columns`
- This creates `perfect multicollinearity`, which can cause problems in statistical models

Why First Column?
- The first column is typically dropped by convention
- It could be any column, but dropping the first one is a common practice
- The dropped category becomes the "reference" or "baseline" category
- All other categories are interpreted relative to this baseline

In [26]:
# Training the encoder
X_train,X_test,y_train,y_test = train_test_split(X, y, test_size=0.2, random_state=42)

ohe = OneHotEncoder(drop='first', sparse_output=False)
ohe.fit(X_train)

,categories,'auto'
,drop,'first'
,sparse_output,False
,dtype,<class 'numpy.float64'>
,handle_unknown,'error'
,min_frequency,None
,max_categories,None
,feature_name_combiner,'concat'


In [27]:
# 0th category from 1st column and 0th category from 2nd column
ohe.drop_idx_ 

array([0, 0], dtype=object)

#### Handling Rare Categories

In [28]:
# Frequency of categories in brand feature
X_train['brand'].value_counts()

brand
Maruti           1953
Hyundai          1127
Mahindra          635
Tata              586
Toyota            391
Honda             369
Ford              320
Chevrolet         185
Renault           183
Volkswagen        154
BMW                96
Skoda              82
Nissan             62
Jaguar             59
Volvo              54
Datsun             48
Mercedes-Benz      43
Fiat               35
Audi               30
Jeep               26
Lexus              22
Mitsubishi         13
Force               6
Land                5
Kia                 4
Daewoo              3
MG                  3
Ambassador          3
Isuzu               2
Ashok               1
Peugeot             1
Opel                1
Name: count, dtype: int64

In [29]:
# Frequency of categories in fuel feature
cars['fuel'].value_counts()

fuel
Diesel    4402
Petrol    3631
CNG         57
LPG         38
Name: count, dtype: int64

In [30]:
# using min frequency
ohe = OneHotEncoder(sparse_output=False, min_frequency=100)
ohe.fit(X_train)

,categories,'auto'
,drop,None
,sparse_output,False
,dtype,<class 'numpy.float64'>
,handle_unknown,'error'
,min_frequency,100
,max_categories,None
,feature_name_combiner,'concat'


In [31]:
print(ohe.transform(X_test).shape[1])
print((X_train['brand'].nunique(), X_train['fuel'].nunique()))

14
(32, 4)


In [32]:
# Frequent categories
ohe.get_feature_names_out()

array(['brand_Chevrolet', 'brand_Ford', 'brand_Honda', 'brand_Hyundai',
       'brand_Mahindra', 'brand_Maruti', 'brand_Renault', 'brand_Tata',
       'brand_Toyota', 'brand_Volkswagen', 'brand_infrequent_sklearn',
       'fuel_Diesel', 'fuel_Petrol', 'fuel_infrequent_sklearn'],
      dtype=object)

In [33]:
# using max_categories
ohe = OneHotEncoder(sparse_output=False, handle_unknown='ignore', max_categories=15)
ohe.fit(X_train)

,categories,'auto'
,drop,None
,sparse_output,False
,dtype,<class 'numpy.float64'>
,handle_unknown,'ignore'
,min_frequency,None
,max_categories,15
,feature_name_combiner,'concat'


In [34]:
print(ohe.transform(X_test).shape[1])
print((X_train['brand'].nunique(), X_train['fuel'].nunique()))

19
(32, 4)


In [35]:
ohe.get_feature_names_out()

array(['brand_BMW', 'brand_Chevrolet', 'brand_Ford', 'brand_Honda',
       'brand_Hyundai', 'brand_Jaguar', 'brand_Mahindra', 'brand_Maruti',
       'brand_Nissan', 'brand_Renault', 'brand_Skoda', 'brand_Tata',
       'brand_Toyota', 'brand_Volkswagen', 'brand_infrequent_sklearn',
       'fuel_CNG', 'fuel_Diesel', 'fuel_LPG', 'fuel_Petrol'], dtype=object)

In [36]:
# Handling unknowk category
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)

# Training
ohe = OneHotEncoder(sparse_output=False, handle_unknown='ignore') # If the category is not present in trained data then all the digits will be encoded by 0
ohe.fit(X_train)

,categories,'auto'
,drop,None
,sparse_output,False
,dtype,<class 'numpy.float64'>
,handle_unknown,'ignore'
,min_frequency,None
,max_categories,None
,feature_name_combiner,'concat'


In [37]:
# Reverse Encoding
ohe.inverse_transform(np.array([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1.]).reshape(1, 36))

array([[None, 'Petrol']], dtype=object)

---

<h4>LabelBinarizer</h4>

This type of encoding is One Hot Encoding of target column.

In [38]:
from sklearn.preprocessing import LabelBinarizer

# Sample target variable for a multi-class classification problem
y = ['cat', 'dog', 'fish', 'dog', 'cat']

# Initialize the LabelBinarizer
lb = LabelBinarizer()

# Fit and transform the target variable
y_binarized = lb.fit_transform(y)

print("Binarized labels:\n", y_binarized)

# Inverse transform to recover original labels
y_original = lb.inverse_transform(y_binarized)

print("\nOriginal labels:\n", y_original)

Binarized labels:
 [[1 0 0]
 [0 1 0]
 [0 0 1]
 [0 1 0]
 [1 0 0]]

Original labels:
 ['cat' 'dog' 'fish' 'dog' 'cat']


Parameters control how the binary labels are encoded when you have a binary classification problem (only two classes).
- neg_label (default = 0): This is the value assigned to the negative class (the first class encountered). By default, it's set to 0
- pos_label (default = 1): This is the value assigned to the positive class (the second class encountered). By default, it's set to 1

In [39]:
# Binary classification example
y = ['cat', 'dog', 'cat', 'dog', 'cat']

# Default behavior (neg_label=0, pos_label=1)
lb_default = LabelBinarizer() # For 2 classes the first class will automatically get dropped
y_default = lb_default.fit_transform(y)
print("Default encoding:\n", y_default)

# Custom labels
lb_custom = LabelBinarizer(neg_label=-1, pos_label=1)
y_custom = lb_custom.fit_transform(y)
print("\n Custom encoding:\n", y_custom)

Default encoding:
 [[0]
 [1]
 [0]
 [1]
 [0]]

 Custom encoding:
 [[-1]
 [ 1]
 [-1]
 [ 1]
 [-1]]


*For 2 classes, the first class will automatically get dropped*

---

#### MultiLabelBinarizer
- MultiLabelBinarizer is used for encoding multiple labels per instance
- It transforms multi-label data into a binary matrix format where each column represents a class
- Useful for scenarios where each sample can belong to multiple categories simultaneously

The possible values of one random variable (red, green, blue). Each instance belongs to one than one values (red, green), (green, blue) ... 

In [40]:
from sklearn.preprocessing import MultiLabelBinarizer

# Example multi-label data
y = [('red', 'blue'), ('blue', 'green'), ('green',), ('red',)]

# Initialize MultiLabelBinarizer
mlb = MultiLabelBinarizer()

# Fit and transform the data to binary matrix format
Y = mlb.fit_transform(y)

print("Binary matrix:\n", Y)
print("\nClass labels:", mlb.classes_)

# Inverse transform to recover original labels
y_inv = mlb.inverse_transform(Y)
print("\nInverse transformed labels:", y_inv)

Binary matrix:
 [[1 0 1]
 [1 1 0]
 [0 1 0]
 [0 0 1]]

Class labels: ['blue' 'green' 'red']

Inverse transformed labels: [('blue', 'red'), ('blue', 'green'), ('green',), ('red',)]
